In [8]:
%%writefile inference.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <chrono>

#define N 512

// ---------------- CPU MATRIX MULTIPLICATION ----------------

void cpu_matmul(float *A, float *B, float *C)
{
    for(int i=0;i<N;i++)
        for(int j=0;j<N;j++)
        {
            float sum = 0;
            for(int k=0;k<N;k++)
                sum += A[i*N+k] * B[k*N+j];

            C[i*N+j] = sum;
        }
}

// ---------------- GPU MATRIX MULTIPLICATION ----------------

__global__ void matmul(float *A, float *B, float *C)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < N && col < N)
    {
        float sum = 0;

        for(int k = 0; k < N; k++)
        {
            sum += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}

// ---------------- RELU ACTIVATION ----------------

__global__ void relu(float *data, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if(i < n)
    {
        if(data[i] < 0)
            data[i] = 0;
    }
}

int main()
{

    size_t size = N * N * sizeof(float);

    float *A = (float*)malloc(size);
    float *B = (float*)malloc(size);
    float *C_cpu = (float*)malloc(size);
    float *C_gpu = (float*)malloc(size);

    for(int i=0;i<N*N;i++)
    {
        A[i] = 1.0f;
        B[i] = 2.0f;
    }

    // ---------------- CPU BENCHMARK ----------------

    auto cpu_start = std::chrono::high_resolution_clock::now();

    cpu_matmul(A,B,C_cpu);

    auto cpu_end = std::chrono::high_resolution_clock::now();

    double cpu_time =
    std::chrono::duration<double>(cpu_end - cpu_start).count();

    printf("CPU Time: %f sec\n",cpu_time);


    // ---------------- GPU MEMORY ----------------

    float *d_A,*d_B,*d_C;

    cudaMalloc(&d_A,size);
    cudaMalloc(&d_B,size);
    cudaMalloc(&d_C,size);

    cudaMemcpy(d_A,A,size,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,B,size,cudaMemcpyHostToDevice);


    dim3 threads(16,16);
    dim3 blocks(N/16,N/16);

    // ---------------- GPU BENCHMARK ----------------

    auto gpu_start = std::chrono::high_resolution_clock::now();

    matmul<<<blocks,threads>>>(d_A,d_B,d_C);

    cudaDeviceSynchronize();

    auto gpu_end = std::chrono::high_resolution_clock::now();

    double gpu_time =
    std::chrono::duration<double>(gpu_end - gpu_start).count();

    printf("GPU Time: %f sec\n",gpu_time);


    cudaMemcpy(C_gpu,d_C,size,cudaMemcpyDeviceToHost);

    // ---------------- RELU ACTIVATION ----------------

    int total = N*N;

    float *d_relu;

    cudaMalloc(&d_relu,size);

    cudaMemcpy(d_relu,C_gpu,size,cudaMemcpyHostToDevice);

    relu<<<(total+255)/256,256>>>(d_relu,total);

    cudaMemcpy(C_gpu,d_relu,size,cudaMemcpyDeviceToHost);

    printf("Sample Output: %f\n",C_gpu[0]);

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);
    cudaFree(d_relu);

    free(A);
    free(B);
    free(C_cpu);
    free(C_gpu);

    return 0;
}

Writing inference.cu


In [9]:
!nvcc inference.cu -o inference

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [10]:
!./inference

CPU Time: 0.641014 sec
GPU Time: 0.021688 sec
Sample Output: 1024.000000
